# Time Domain: Trapping Square with Discontinuous Galerkin

This notebook solves the driven trapping-square problem from the preceding time-domain example with a discontinuous Galerkin discretization. The geometry, hyperboloidal layer, source frequency, and Newmark time integrator are unchanged in spirit. The purpose is not to rank DG against conforming elements, but to expose a discretization issue that continuity hides: the lower-order derivative terms generated by the conformal rescaling and the mixed time-space term require consistent face contributions.

We use a second-order-in-time formulation with an $L^2$ discontinuous space, symmetric interior penalty for the spatial operator, and an energy-balanced skew flux for the mixed term. The sound-soft wall is imposed weakly. The compactified outer boundary is an outflow boundary and receives no artificial boundary condition.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np
import time

notebook_start = time.perf_counter()

## Continuous Operator and the Two Mixed Structures

The compactified weak equation has the semidiscrete form

$$
M\ddot u+C\dot u+Ku=f.
$$

Let $X=(x,y)$, $\rho=|X|$, $Q=XX^T/\rho^2$, $P=I-Q$, and $A=GQ+LP$. Define the regular vector

$$
g=\frac{\Omega\Omega'}{2L\rho}X.
$$

The spatial volume form can be written as

$$
K(u,v)=\int_{\mathcal D}
\left[\nabla u\cdot A\nabla v+g\cdot\nabla u\,v+g\cdot\nabla v\,u+\frac{(\Omega')^2}{4L}uv\right]dx.
$$

The first three terms come from the factored conformal derivative. Its natural flux is therefore not merely $A\nabla u$, but

$$
\mathcal F(u)=A\nabla u+g u.
$$

Using $A\nabla u$ alone in the SIPG face terms would discretize a different global operator when $g$ changes across the layer interface.

The time-space term has a different structure. With $b=HX/\rho$,

$$
C(u,v)=\int_{\mathcal D}\left[v\,b\cdot\nabla u-u\,b\cdot\nabla v\right]dx
+\int_{\mathscr I^+}uv\,ds.
$$

Its interior part is skew. The DG skeleton term below preserves that cancellation across discontinuous element traces; the remaining quadratic contribution is the physical outflow term at null infinity.

In [ ]:
# Geometry
Rout = 2.0
RScat = 1.0
DScat = 0.1
DHole = 0.2
R = 1.5
xOffset = 0.5

# DG discretization
maxh = 0.20
order = 3
penalty = 8.0

# Time integration and source
dt = 2e-2
beta = 1 / 4
gamma = 1 / 2
drive_frequency = 10.8
drive_until = 3.0
t_end = 6.0
source_amplitude = 100.0

In [ ]:
def create_trapping_geometry(draw=False):
    domain = Circle((0, 0), Rout).Face()
    domain.edges.name = "outer"
    domain.faces.name = "outer"

    inner = Circle((0, 0), R).Face()
    inner.edges.name = "interface"
    inner.faces.name = "inner"

    wall = (
        MoveTo(-RScat / 2 + xOffset, -RScat / 2).Rectangle(RScat, RScat).Face()
        - MoveTo(-RScat / 2 + DScat / 2 + xOffset, -RScat / 2 + DScat / 2)
          .Rectangle(RScat - DScat, RScat - DScat).Face()
        - MoveTo(-RScat / 2 - DScat + xOffset, -DHole / 2)
          .Rectangle(2 * DScat, DHole).Face()
    )
    wall.edges.name = "scatterer"

    cavity = (
        MoveTo(-RScat / 2 + xOffset, -RScat / 2).Rectangle(RScat, RScat).Face()
        - wall
    )
    cavity.faces.name = "cavity"

    pieces = Glue([domain - inner, inner - cavity - wall, cavity])
    if draw:
        Draw(pieces)
    return OCCGeometry(pieces, dim=2)


mesh = Mesh(create_trapping_geometry().GenerateMesh(maxh=maxh))
mesh.Curve(order)
fes = L2(mesh, order=order, dgjumps=True)
print(f"DG degrees of freedom: {fes.ndof}")

## SIPG Form with the Conformal Flux

On an interior face $F$, write $[u]=u^- -u^+$ and let braces denote an average. The consistency part of the SIPG form is

$$
-\int_F\{\mathcal F(u)\cdot n\}[v],ds
-\int_F\{\mathcal F(v)\cdot n\}[u],ds,
$$

followed by the usual penalty proportional to $(p+1)^2(n^TAn)/h$. The use of $\mathcal F=A\nabla+g$ is the important lower-order choice. It keeps the face terms tied to the factored continuous operator instead of treating the $g\cdot\nabla$ pieces as unrelated elementwise reactions.

The same flux is used in the weak sound-soft boundary terms. At $\mathscr I^+$ the radial diffusivity vanishes, so no SIPG boundary condition is imposed.

In [ ]:
def assemble_dg_forms(mesh, fes):
    u, v = fes.TnT()
    rho = sqrt(x**2 + y**2)
    radial = CF((x, y))

    Omega = mesh.MaterialCF({"outer": (Rout - rho) / (Rout - R)}, default=1)
    DOmega = Omega.Diff(rho)
    L = Omega - rho * DOmega
    G = Omega**2 / L
    H = mesh.MaterialCF({"outer": 1 - G}, default=0)
    mass_weight = 1 + H  # (1-H**2)/G for H=1-G

    Q = OuterProduct(radial, radial) / rho**2
    P = Id(2) - Q
    A = G * Q + L * P
    conformal_vector = Omega / L / (2 * rho) * DOmega * radial

    normal = specialcf.normal(mesh.dim)
    mesh_size = specialcf.mesh_size
    jump_u = u - u.Other()
    jump_v = v - v.Other()

    # The lower-order conformal term belongs in the consistency flux.
    flux_vector_u = A * grad(u) + conformal_vector * u
    flux_vector_v = A * grad(v) + conformal_vector * v
    flux_vector_u_other = A.Other() * grad(u.Other()) + conformal_vector.Other() * u.Other()
    flux_vector_v_other = A.Other() * grad(v.Other()) + conformal_vector.Other() * v.Other()
    average_flux_u = 0.5 * (flux_vector_u + flux_vector_u_other)
    average_flux_v = 0.5 * (flux_vector_v + flux_vector_v_other)

    normal_diffusion = InnerProduct(normal, A * normal)
    normal_diffusion_other = InnerProduct(normal, A.Other() * normal)
    average_normal_diffusion = 0.5 * (normal_diffusion + normal_diffusion_other)
    average_h = 0.5 * (mesh_size + mesh_size.Other())
    sigma = penalty * (order + 1) ** 2 * average_normal_diffusion / average_h

    stiffness_volume = (
        InnerProduct(grad(u), A * grad(v))
        + InnerProduct(conformal_vector, grad(u)) * v
        + InnerProduct(conformal_vector, grad(v)) * u
        + DOmega**2 / (4 * L) * u * v
    ) * dx
    stiffness_faces = (
        -InnerProduct(average_flux_u, normal) * jump_v
        -InnerProduct(average_flux_v, normal) * jump_u
        + sigma * jump_u * jump_v
    ) * dx(skeleton=True)

    # Weak homogeneous Dirichlet condition on the sound-soft wall.
    boundary_sigma = penalty * (order + 1) ** 2 * normal_diffusion / mesh_size
    stiffness_wall = (
        -InnerProduct(flux_vector_u, normal) * v
        -InnerProduct(flux_vector_v, normal) * u
        + boundary_sigma * u * v
    ) * ds("scatterer", skeleton=True)

    # Skew volume-plus-skeleton form for the mixed time-space term.
    transport_vector = H / rho * radial
    average_transport = 0.5 * (transport_vector + transport_vector.Other())
    normal_transport = InnerProduct(average_transport, normal)
    transport = (
        -u * InnerProduct(transport_vector, grad(v)) * dx
        + v * InnerProduct(transport_vector, grad(u)) * dx
        + normal_transport * (u.Other() * v - u * v.Other()) * dx(skeleton=True)
        + u * v * ds("outer", skeleton=True)
    )

    mass = mass_weight * u * v * dx
    stiffness = stiffness_volume + stiffness_faces + stiffness_wall
    return mass, transport, stiffness


mass_form, transport_form, stiffness_form = assemble_dg_forms(mesh, fes)

## Why the Mixed Skeleton Term Matters

For a conforming function, the interior-face term in $C_h$ vanishes because the two traces agree. For a DG function it supplies the trace coupling produced when the $-u\,b\cdot\nabla v$ term is integrated by parts element by element. The face term is itself skew: exchanging trial and test functions changes its sign. Consequently the interior volume and face contributions make no quadratic contribution, and only

$$
C_h(w,w)=\int_{\mathscr I^+}w^2\,ds
$$

remains. This is the outgoing energy flux. There is a subtle trap here: omitting the skeleton term does *not* necessarily reveal itself in $C_h(w,w)$, because the elementwise volume pair is already skew. It instead removes the interelement transport and fails the consistency argument for different trial and test traces. An energy check alone is therefore insufficient.

The fitted physical-layer interface needs no special hyperboloidal penalty. It is an ordinary DG face evaluated with the coefficients and traces from both sides.

## Newmark Evolution

The average-acceleration Newmark update is the same as in the conforming notebook. With $\beta=1/4$ and $\gamma=1/2$, every acceleration update uses

$$
S^{-1}=\left(M+\gamma\Delta t C+\beta\Delta t^2K\right)^{-1}.
$$

The assembled source vector is acted on by this effective inverse. We record a phase-space $L^2$ magnitude in the unchanged physical region and inside the cavity; it is a diagnostic, not a discrete-energy identity. For visualization, the evolution cell also stores one DG snapshot every $0.2$ time units in a multidimensional grid function. This sampling does not alter the Newmark update.

In [ ]:
assembly_start = time.perf_counter()
M = BilinearForm(mass_form).Assemble().mat
C = BilinearForm(transport_form).Assemble().mat
K = BilinearForm(stiffness_form).Assemble().mat
Sinv = BilinearForm(
    mass_form + gamma * dt * transport_form + beta * dt**2 * stiffness_form
).Assemble().mat.Inverse()
Minv = M.Inverse()
assembly_seconds = time.perf_counter() - assembly_start

u, v = fes.TnT()
outer_flux_matrix = BilinearForm(
    u * v * ds("outer", skeleton=True),
    check_unused=False,
).Assemble().mat
probe = GridFunction(fes)
probe.vec.FV().NumPy()[:] = np.random.default_rng(7).standard_normal(fes.ndof)
mixed_quadratic = probe.vec.InnerProduct(C * probe.vec).real
outflow_quadratic = probe.vec.InnerProduct(outer_flux_matrix * probe.vec).real
mixed_identity_residual = abs(mixed_quadratic - outflow_quadratic) / abs(outflow_quadratic)

source_profile = exp(-30 * ((x + 0.8) ** 2 + (y + 0.8) ** 2))
force = LinearForm(
    source_amplitude * source_profile * v * dx("inner|cavity")
).Assemble().vec
effective_force = Sinv * force

solution = GridFunction(fes, name="u_dg")
velocity = GridFunction(fes, name="ut_dg")
acceleration = GridFunction(fes, name="utt_dg")
acceleration.vec.data = -Minv @ K * solution.vec - Minv @ C * velocity.vec
old_acceleration = acceleration.vec.CreateVector()

physical_mass = BilinearForm(
    u * v * dx("inner|cavity"), check_unused=False
).Assemble().mat
cavity_mass = BilinearForm(
    u * v * dx("cavity"), check_unused=False
).Assemble().mat

sample_times = []
physical_magnitudes = []
cavity_magnitudes = []
animation = GridFunction(fes, multidim=0, name="u_dg_history")
frame_stride = max(1, int(round(0.2 / dt)))
evolution_start = time.perf_counter()
nsteps = int(round(t_end / dt))

for step in range(nsteps):
    t = step * dt
    if step % frame_stride == 0:
        physical_q = solution.vec.InnerProduct(physical_mass * solution.vec).real
        physical_v = velocity.vec.InnerProduct(physical_mass * velocity.vec).real
        cavity_q = solution.vec.InnerProduct(cavity_mass * solution.vec).real
        cavity_v = velocity.vec.InnerProduct(cavity_mass * velocity.vec).real
        sample_times.append(t)
        physical_magnitudes.append(np.sqrt(max(0, physical_q + physical_v / drive_frequency**2)))
        cavity_magnitudes.append(np.sqrt(max(0, cavity_q + cavity_v / drive_frequency**2)))
        animation.AddMultiDimComponent(solution.vec)

    old_acceleration.data = acceleration.vec
    acceleration.vec.data = (
        -Sinv @ K * solution.vec
        -Sinv @ C * velocity.vec
        -dt * Sinv @ K * velocity.vec
        + (gamma - 1) * dt * Sinv @ C * old_acceleration
        + (beta - 0.5) * dt**2 * Sinv @ K * old_acceleration
        + ((t + dt) < drive_until)
          * np.sin(drive_frequency * (t + dt))
          * effective_force
    )
    solution.vec.data += (
        dt * velocity.vec
        + (0.5 - beta) * dt**2 * old_acceleration
        + beta * dt**2 * acceleration.vec
    )
    velocity.vec.data += (
        (1 - gamma) * dt * old_acceleration
        + gamma * dt * acceleration.vec
    )

animation.AddMultiDimComponent(solution.vec)  # include the state at t=t_end
evolution_seconds = time.perf_counter() - evolution_start
sample_times = np.asarray(sample_times)
physical_magnitudes = np.asarray(physical_magnitudes)
cavity_magnitudes = np.asarray(cavity_magnitudes)
print(f"mixed-form outflow identity residual: {mixed_identity_residual:.3e}")
print(f"matrix assembly and factorization: {assembly_seconds:.3f} s")
print(f"Newmark evolution ({nsteps} steps): {evolution_seconds:.3f} s")

In [ ]:
Draw(
    animation, mesh, "DG compactified time evolution",
    min=-0.1, max=0.1, animate=True,
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(sample_times, physical_magnitudes, label="physical region")
ax.semilogy(sample_times, cavity_magnitudes, label="cavity")
ax.axvline(drive_until, color="0.4", ls="--", label="source switched off")
ax.set(xlabel="time", ylabel="phase-space $L^2$ magnitude")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()
print(f"notebook compute time after imports: {time.perf_counter() - notebook_start:.3f} s")

## Interpretation and Scope

The animation shows the compactified DG unknown at intervals of $0.2$ time units. It agrees with the physical field inside the layer interface and remains regular at null infinity. The source excites the physical region and is switched off at the marked time. Radiation then leaves through the hyperboloidal layer while a component remains in the trapping cavity. The calculation demonstrates that the compactified boundary and fitted layer interface can be handled with standard DG face machinery once the correct continuous flux structure is retained.

This notebook uses an energy-balanced central treatment of the mixed operator together with SIPG stabilization of spatial jumps. Upwind or Rusanov dissipation is more naturally introduced after a first-order characteristic reduction. The first-order and second-order approaches are algebraically related, but their fluxes should not be mixed ad hoc.

The main lesson is structural: preserve conservative and skew pairs before discretization. Expanding variable-coefficient derivatives into separate lower-order terms can hide interface distributions and lead to a DG method that is elementwise plausible but globally inconsistent.